# 03 · FT-Transformer — Custom Numerical Embedding Experiments (Lending Club)

Comparison of four custom replacements for the default linear numerical embedding in FT-Transformer:

1. **Linear + PLE** — concatenates a linear embedding with a Piecewise    Linear Encoding (quantile-bin based) for each continuous feature.
2. **MLP-based** — a small MLP mixes feature values into context    before per-feature embedding.
3. **Cross-SwiGLU** — a cross-feature SwiGLU tokenizer that breaks    the per-feature independence assumption of vanilla FTT.
4. **Periodic** — periodic (sin/cos) feature embedding from the    Rubachev et al. paper.

All four variants share the same dataset, preprocessing, training loop and hyperparameters so their validation/test AUCs are directly comparable.

## 1. Setup

In [ ]:
# %pip install rtdl_revisiting_models -q
%pip install delu -q
%pip install optuna -q

In [ ]:
import math
import random
import warnings
import json
import os
import itertools
import typing
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import numpy as np
import pandas as pd
import scipy.special

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter
from torch.utils.data import Dataset, DataLoader, TensorDataset

import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import sklearn.tree as sklearn_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

import optuna
import delu
from tqdm.std import tqdm
from tqdm import tqdm as _tqdm  # alias for cells that use it directly

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 200
pd.options.display.max_columns = 200

RANDOM_SEED = 42


def seed_everything(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# --- Optional: Google Colab Drive mount ---
# Uncomment the three lines below if you're running on Colab and want to read
# data from / write artifacts to your Drive. Skip on local, server, or Kaggle
# runs. The next cell auto-routes through DRIVE_ROOT when defined.
#
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ROOT = "/content/drive/MyDrive/ft-transformer-credit-risk-study"

# When running locally, repo root is one directory up (notebook is in
# notebooks/<dataset>/). On Colab with the cell above uncommented, DRIVE_ROOT
# takes precedence.
_BASE = globals().get("DRIVE_ROOT", "..")
DATA_PATH      = f"{_BASE}/data/processed_data_densest.parquet.gzip"
ARTIFACTS_DIR  = Path(f"{_BASE}/artifacts/lending_club")
RESULTS_DIR    = ARTIFACTS_DIR / "results"
MODELS_DIR     = ARTIFACTS_DIR / "models"
FIGURES_DIR    = ARTIFACTS_DIR / "figures"
for _d in (ARTIFACTS_DIR, RESULTS_DIR, MODELS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH      = {DATA_PATH}")
print(f"ARTIFACTS_DIR  = {ARTIFACTS_DIR}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# DEV_MODE — smoke-test switch.
#
# When False (the default), every constant resolves to the exact value used in
# the original experiment. The DEV_MODE branches below are dead code in that
# path, so the run is behaviourally identical to the source notebook.
#
# When True, the notebook runs a tiny fast version: a subsample of the data,
# a couple of epochs, a single Optuna trial. Use this on Colab to validate the
# full pipeline end-to-end before launching a real run.
# ──────────────────────────────────────────────────────────────────────────────
DEV_MODE = False
if DEV_MODE:
    print("DEV_MODE is ON — smoke-test run with reduced data/epochs/trials.")

## 2. Data Loading

In [ ]:
# Load preprocessed Lending Club dataset.
df4 = pd.read_parquet(DATA_PATH)

train_val_df, test_df = train_test_split(
    df4, test_size=0.2,
    stratify=df4["target_binary"], random_state=RANDOM_SEED,
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.2,
    stratify=train_val_df["target_binary"], random_state=RANDOM_SEED,
)
print("Train_val / test sizes:", len(train_val_df), len(test_df))
print("Train / val sizes:", len(train_df), len(val_df))

cat_columns = [
    "term", "emp_length", "home_ownership", "verification_status",
    "purpose", "zip_code", "addr_state", "application_type",
    "initial_list_status", "disbursement_method",
]
num_cols = [
    c for c in list(df4.columns)
    if c not in cat_columns + ["id", "emp_title", "target_binary"]
]
print(f"# numerical features: {len(num_cols)}  |  # categorical: {len(cat_columns)}")

## 3. Preprocessing

In [ ]:
# Numeric imputation + scaling (median + StandardScaler).
num_impute = {}
scaler = StandardScaler()

num_df = train_df[num_cols].copy()
for c in num_df.columns:
    if num_df[c].dtype == "object":
        num_df[c] = pd.to_numeric(num_df[c], errors="coerce")
    med = num_df[c].median()
    num_df[c] = num_df[c].fillna(med)
    num_impute[c] = med
num_df[num_df.columns] = scaler.fit_transform(num_df[num_df.columns])

# Label encoding with a permanent UNKNOWN bucket for unseen validation/test categories.
cat_encoders: Dict[str, LabelEncoder] = {}
cat_cardinalities = []
cat_df = train_df[cat_columns].copy()
for c in cat_columns:
    le = LabelEncoder()
    series = cat_df[c].fillna("MISSING").astype(str)
    le.fit(series)
    new_classes = list(le.classes_)
    if "MISSING" not in new_classes:
        new_classes.append("MISSING")
    new_classes.append("UNKNOWN")
    le.classes_ = np.array(new_classes)
    cat_df[c + "_le"] = le.transform(series)
    cat_encoders[c] = le
    cat_cardinalities.append(len(le.classes_))

train_df_proc = pd.concat(
    [
        num_df.reset_index(drop=True),
        cat_df[[c + "_le" for c in cat_columns]].reset_index(drop=True),
        train_df["target_binary"].reset_index(drop=True),
    ],
    axis=1,
)
print("Encoded train")

# Apply the same imputation, scaling and label maps to val and test.
def _apply_transforms(part_df):
    num_part = part_df[num_cols].copy()
    for c in num_part.columns:
        if num_part[c].dtype == "object":
            num_part[c] = pd.to_numeric(num_part[c], errors="coerce")
    for c in num_part.columns:
        num_part[c] = num_part[c].fillna(num_impute[c])
    num_part = pd.DataFrame(
        scaler.transform(num_part), columns=num_part.columns, index=num_part.index
    )

    cat_part = part_df[cat_columns].copy()
    for c in cat_columns:
        le = cat_encoders[c]
        mapping = {cls: i for i, cls in enumerate(le.classes_)}
        unknown_id = mapping["UNKNOWN"]
        series = cat_part[c].fillna("MISSING").astype(str)
        cat_part[c + "_le"] = series.map(mapping).fillna(unknown_id).astype(int)

    return pd.concat(
        [
            num_part.reset_index(drop=True),
            cat_part[[c + "_le" for c in cat_columns]].reset_index(drop=True),
            part_df["target_binary"].reset_index(drop=True),
        ],
        axis=1,
    )


val_df_proc = _apply_transforms(val_df)
test_df_proc = _apply_transforms(test_df)
print("Encoded val and test")

In [ ]:
# Materialize everything as torch tensors on the target device.
cat_le_cols = [c + "_le" for c in cat_columns]
data_numpy = {"train": {}, "val": {}, "test": {}}
for part_name, part_df in [
    ("train", train_df_proc),
    ("val",   val_df_proc),
    ("test",  test_df_proc),
]:
    data_numpy[part_name]["X_cont"] = part_df[num_cols]
    data_numpy[part_name]["x_cat"]  = part_df[cat_le_cols]
    data_numpy[part_name]["y"]      = part_df["target_binary"]

data = {}
for part in data_numpy:
    data[part] = {
        "X_cont": torch.tensor(data_numpy[part]["X_cont"].to_numpy(), dtype=torch.float32, device=device),
        "x_cat":  torch.tensor(data_numpy[part]["x_cat"].to_numpy(),  dtype=torch.long,    device=device),
        "y":      torch.tensor(data_numpy[part]["y"].to_numpy(),      dtype=torch.float32, device=device),
    }

d_numerical = len(num_cols)
n_train = data["train"]["y"].shape[0]
print(f"d_numerical = {d_numerical}")
print(f"cat_cardinalities = {cat_cardinalities}")
print(f"# train rows = {n_train}")

In [ ]:
# When DEV_MODE is on, subsample data to a few thousand rows so a full training
# pass completes in a couple of minutes. No-op when DEV_MODE is False.
if DEV_MODE:
    _n_dev = 5000
    for _part in data:
        _idx = torch.randperm(data[_part]["y"].shape[0], device=device)[:_n_dev]
        for _k in data[_part]:
            data[_part][_k] = data[_part][_k][_idx]
    print({_part: {k: v.shape for k, v in data[_part].items()} for _part in data})

## 4. Quantile Boundary Helper (used by PLE variants)

In [ ]:
def compute_quantile_boundaries(X_train, n_bins=10):
    """
    X_train: numpy array or torch tensor of shape [n_samples, n_features]
    Returns: torch tensor of shape [n_features, n_bins + 1]
    """
    if torch.is_tensor(X_train):
        X_train = X_train.detach().cpu().numpy()
        
    n_features = X_train.shape[1]
    boundaries = []
    
    # Generate bin edges for each feature independently
    quantiles = np.linspace(0, 1, n_bins + 1)
    
    for i in range(n_features):
        feature_column = X_train[:, i]
        # Use unique to handle heavily clipped or zero-inflated data
        b = np.unique(np.quantile(feature_column, quantiles))
        
        # If unique values < n_bins, pad with the last value
        if len(b) < n_bins + 1:
            b = np.concatenate([b, np.full(n_bins + 1 - len(b), b[-1])])
            
        boundaries.append(b)
        
    return torch.tensor(np.array(boundaries), dtype=torch.float32)

boundaries_df = compute_quantile_boundaries(data['train']['X_cont'], n_bins=10)

## 5. Variant 1 — Linear + PLE Embedding

Concatenates a standard linear embedding with a Quantile-based Piecewise Linear Encoding. The linear branch acts as a residual; the PLE branch captures the empirical CDF of each feature.

### 5.1 Model

In [ ]:
import itertools
import math
import typing
import warnings
from collections import OrderedDict
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter

_INTERNAL_ERROR = 'Internal error'
import numpy as np
import torch

def _named_sequential(*modules) -> nn.Sequential:
    return nn.Sequential(OrderedDict(modules))

class PeriodicEmbedding(nn.Module):
    def __init__(self, n_features, d_token, sigma=1.0):
        super().__init__()
        # Coefficients are learned; initialized with a normal distribution
        self.coefficients = nn.Parameter(torch.randn(n_features, d_token // 2) * sigma)
        
    def forward(self, x):
        # x shape: [batch_size, n_features]
        # We want: [batch_size, n_features, d_token]
        x = x.unsqueeze(-1) # [B, N, 1]
        
        # Element-wise multiplication with broadcasting
        # [B, N, 1] * [N, d_token/2] -> [B, N, d_token/2]
        out = 2 * math.pi * self.coefficients * x
        
        # Concatenate sine and cosine for the final embedding
        return torch.cat([torch.cos(out), torch.sin(out)], dim=-1)

class QuantilePLEEmbedding(nn.Module):
    def __init__(self, boundaries, d_token):
        """
        boundaries: Tensor of shape [n_features, n_bins + 1]
        """
        super().__init__()
        self.n_features, n_bins_plus_one = boundaries.shape
        self.n_bins = n_bins_plus_one - 1
        self.d_token = d_token
        
        # Store boundaries so they aren't updated by gradients
        self.register_buffer("boundaries", boundaries)
        
        # Learnable embeddings for each bin of each feature
        # Shape: [n_features, n_bins, d_token]
        self.embeddings = nn.Parameter(torch.randn(self.n_features, self.n_bins, d_token) * 0.02)

    def forward(self, x):
        # x: [B, n_features]
        B = x.shape[0]
        
        # 1. Expand x and boundaries for broadcasting
        # x: [B, n_features, 1]
        x_ext = x.unsqueeze(-1)
        
        # boundaries: [1, n_features, n_bins + 1]
        L = self.boundaries[:, :-1].unsqueeze(0)
        R = self.boundaries[:, 1:].unsqueeze(0)
        
        # 2. Calculate relative fill of each bin (0.0 to 1.0)
        # Using clamp to ensure values below L are 0 and above R are 1
        # This creates the "Piecewise" effect
        widths = R - L
        # Small epsilon to avoid division by zero in identical quantiles
        widths = torch.clamp(widths, min=1e-6)
        
        weights = (x_ext - L) / widths
        weights = torch.clamp(weights, 0.0, 1.0) # [B, n_features, n_bins]
        
        # 3. Multiply weights by bin-specific embeddings
        # [B, n_features, n_bins, 1] * [1, n_features, n_bins, d_token]
        out = weights.unsqueeze(-1) * self.embeddings.unsqueeze(0)
        
        # 4. Sum across bins to get the final token for each feature
        return out.sum(dim=2) # Result: [B, n_features, d_token]
    
class LinearEmbeddings(nn.Module):
    def __init__(self, n_features: int, d_embedding: int) -> None:
        """
        Args:
            n_features: the number of continous features.
            d_embedding: the embedding size.
        """
        if n_features <= 0:
            raise ValueError(f'n_features must be positive, however: {n_features=}')
        if d_embedding <= 0:
            raise ValueError(f'd_embedding must be positive, however: {d_embedding=}')

        super().__init__()
        self.weight = Parameter(torch.empty(n_features, d_embedding))
        self.bias = Parameter(torch.empty(n_features, d_embedding))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rqsrt = self.weight.shape[1] ** -0.5
        nn.init.uniform_(self.weight, -d_rqsrt, d_rqsrt)
        nn.init.uniform_(self.bias, -d_rqsrt, d_rqsrt)

    def forward(self, x: Tensor) -> Tensor:
        if x.ndim < 2:
            raise ValueError(
                f'The input must have at least two dimensions, however: {x.ndim=}'
            )

        x = x[..., None] * self.weight
        x = x + self.bias[None]
        return x


class CategoricalEmbeddings(nn.Module):
    def __init__(
        self, cardinalities: List[int], d_embedding: int, bias: bool = True
    ) -> None:
        super().__init__()
        if not cardinalities:
            raise ValueError('cardinalities must not be empty')
        if any(x <= 0 for x in cardinalities):
            i, value = next((i, x) for i, x in enumerate(cardinalities) if x <= 0)
            raise ValueError(
                'cardinalities must contain only positive values,'
                f' however: cardinalities[{i}]={value}'
            )
        if d_embedding <= 0:
            raise ValueError(f'd_embedding must be positive, however: {d_embedding=}')

        self.embeddings = nn.ModuleList(
            [nn.Embedding(x, d_embedding) for x in cardinalities]
        )
        self.bias = (
            Parameter(torch.empty(len(cardinalities), d_embedding)) if bias else None
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rsqrt = self.embeddings[0].embedding_dim ** -0.5
        for m in self.embeddings:
            nn.init.uniform_(m.weight, -d_rsqrt, d_rsqrt)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -d_rsqrt, d_rsqrt)

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass."""
        if x.ndim < 2:
            raise ValueError(
                f'The input must have at least two dimensions, however: {x.ndim=}'
            )
        n_features = len(self.embeddings)
        if x.shape[-1] != n_features:
            raise ValueError(
                'The last input dimension (the number of categorical features) must be'
                ' equal to the number of cardinalities passed to the constructor.'
                f' However: {x.shape[-1]=}, len(cardinalities)={n_features}'
            )

        x = torch.stack(
            [self.embeddings[i](x[..., i]) for i in range(n_features)], dim=-2
        )
        if self.bias is not None:
            x = x + self.bias
        return x


_LINFORMER_KV_COMPRESSION_SHARING = Literal['headwise', 'key-value']


class MultiheadAttention(nn.Module):
    def __init__(
        self,
        *,
        d_embedding: int,
        n_heads: int,
        dropout: float,
        # Linformer arguments.
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ) -> None:
        if n_heads < 1:
            raise ValueError(f'n_heads must be positive, however: {n_heads=}')
        if d_embedding % n_heads:
            raise ValueError(
                'd_embedding must be a multiple of n_heads,'
                f' however: {d_embedding=}, {n_heads=}'
            )

        super().__init__()
        self.W_q = nn.Linear(d_embedding, d_embedding)
        self.W_k = nn.Linear(d_embedding, d_embedding)
        self.W_v = nn.Linear(d_embedding, d_embedding)
        self.W_out = nn.Linear(d_embedding, d_embedding) if n_heads > 1 else None
        self.dropout = nn.Dropout(dropout) if dropout else None
        self._n_heads = n_heads

        if linformer_kv_compression_ratio is not None:
            if n_tokens is None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is not None,'
                    ' then n_tokens also must not be None'
                )
            if linformer_kv_compression_sharing not in typing.get_args(
                _LINFORMER_KV_COMPRESSION_SHARING
            ):
                raise ValueError(
                    'Valid values of linformer_kv_compression_sharing include:'
                    f' {typing.get_args(_LINFORMER_KV_COMPRESSION_SHARING)},'
                    f' however: {linformer_kv_compression_sharing=}'
                )
            if (
                linformer_kv_compression_ratio <= 0.0
                or linformer_kv_compression_ratio >= 1.0
            ):
                raise ValueError(
                    'linformer_kv_compression_ratio must be from the open interval'
                    f' (0.0, 1.0), however: {linformer_kv_compression_ratio=}'
                )

            def make_linformer_kv_compression():
                return nn.Linear(
                    n_tokens,
                    max(int(n_tokens * linformer_kv_compression_ratio), 1),
                    bias=False,
                )

            self.key_compression = make_linformer_kv_compression()
            self.value_compression = (
                make_linformer_kv_compression()
                if linformer_kv_compression_sharing == 'headwise'
                else None
            )
        else:
            if n_tokens is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then n_tokens also must be None'
                )
            if linformer_kv_compression_sharing is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then linformer_kv_compression_sharing also must be None'
                )
            self.key_compression = None
            self.value_compression = None

        for m in [self.W_q, self.W_k, self.W_v]:
            nn.init.zeros_(m.bias)
        if self.W_out is not None:
            nn.init.zeros_(self.W_out.bias)

    def _reshape(self, x: Tensor) -> Tensor:
        batch_size, n_tokens, d = x.shape
        d_head = d // self._n_heads
        return (
            x.reshape(batch_size, n_tokens, self._n_heads, d_head)
            .transpose(1, 2)
            .reshape(batch_size * self._n_heads, n_tokens, d_head)
        )

    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        """Do the forward pass."""
        q, k, v = self.W_q(x_q), self.W_k(x_kv), self.W_v(x_kv)
        if self.key_compression is not None:
            k = self.key_compression(k.transpose(1, 2)).transpose(1, 2)
            v = (
                self.key_compression
                if self.value_compression is None
                else self.value_compression
            )(v.transpose(1, 2)).transpose(1, 2)

        batch_size = len(q)
        d_head_key = k.shape[-1] // self._n_heads
        d_head_value = v.shape[-1] // self._n_heads
        n_q_tokens = q.shape[1]

        q = self._reshape(q)
        k = self._reshape(k)
        attention_logits = q @ k.transpose(1, 2) / math.sqrt(d_head_key)
        attention_probs = F.softmax(attention_logits, dim=-1)
        if self.dropout is not None:
            attention_probs = self.dropout(attention_probs)
        x = attention_probs @ self._reshape(v)
        x = (
            x.reshape(batch_size, self._n_heads, n_q_tokens, d_head_value)
            .transpose(1, 2)
            .reshape(batch_size, n_q_tokens, self._n_heads * d_head_value)
        )
        if self.W_out is not None:
            x = self.W_out(x)
        return x

class _ReGLU(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        if x.shape[-1] % 2:
            raise ValueError(
                'For the ReGLU activation, the last input dimension'
                f' must be a multiple of 2, however: {x.shape[-1]=}'
            )
        a, b = x.chunk(2, dim=-1)
        return a * F.relu(b)


_TransformerFFNActivation = Literal['ReLU', 'ReGLU']

class _CLSEmbedding(nn.Module):
    def __init__(self, d_embedding: int) -> None:
        super().__init__()
        self.weight = Parameter(torch.empty(d_embedding))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rsqrt = self.weight.shape[-1] ** -0.5
        nn.init.uniform_(self.weight, -d_rsqrt, d_rsqrt)

    def forward(self, batch_dims: Tuple[int]) -> Tensor:
        if not batch_dims:
            raise ValueError('The input must be non-empty')

        return self.weight.expand(*batch_dims, 1, -1)

class FTTransformerBackbone(nn.Module):
    def __init__(
        self,
        *,
        d_out: Optional[int],
        n_blocks: int,
        d_block: int,
        attention_n_heads: int,
        attention_dropout: float,
        ffn_d_hidden: Optional[int] = None,
        ffn_d_hidden_multiplier: Optional[float],
        ffn_dropout: float,
        # NOTE[DIFF]
        # In the paper, FT-Transformer uses the ReGLU activation.
        # Here, to illustrate the difference, ReLU activation is also supported
        # (in particular, see the docstring).
        ffn_activation: _TransformerFFNActivation = 'ReGLU',
        residual_dropout: float,
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ):
        if ffn_activation not in typing.get_args(_TransformerFFNActivation):
            raise ValueError(
                'ffn_activation must be one of'
                f' {typing.get_args(_TransformerFFNActivation)}.'
                f' However: {ffn_activation=}'
            )
        if ffn_d_hidden is None:
            if ffn_d_hidden_multiplier is None:
                raise ValueError(
                    'If ffn_d_hidden is None,'
                    ' then ffn_d_hidden_multiplier must not be None'
                )
            ffn_d_hidden = int(d_block * cast(float, ffn_d_hidden_multiplier))
        else:
            if ffn_d_hidden_multiplier is not None:
                raise ValueError(
                    'If ffn_d_hidden is not None,'
                    ' then ffn_d_hidden_multiplier must be None'
                )
        super().__init__()
        ffn_use_reglu = ffn_activation == 'ReGLU'
        self.blocks = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        # >>> attention
                        'attention': MultiheadAttention(
                            d_embedding=d_block,
                            n_heads=attention_n_heads,
                            dropout=attention_dropout,
                            n_tokens=n_tokens,
                            linformer_kv_compression_ratio=linformer_kv_compression_ratio,
                            linformer_kv_compression_sharing=linformer_kv_compression_sharing,
                        ),
                        'attention_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> feed-forward
                        'ffn_normalization': nn.LayerNorm(d_block),
                        'ffn': _named_sequential(
                            (
                                'linear1',
                                # ReGLU divides dimension by 2,
                                # so multiplying by 2 to compensate for this.
                                nn.Linear(
                                    d_block, ffn_d_hidden * (2 if ffn_use_reglu else 1)
                                ),
                            ),
                            ('activation', _ReGLU() if ffn_use_reglu else nn.ReLU()),
                            ('dropout', nn.Dropout(ffn_dropout)),
                            ('linear2', nn.Linear(ffn_d_hidden, d_block)),
                        ),
                        'ffn_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> output (for hook-based introspection)
                        'output': nn.Identity(),
                        # >>> the very first normalization
                        **(
                            {}
                            if layer_idx == 0
                            else {'attention_normalization': nn.LayerNorm(d_block)}
                        ),
                    }
                )
                for layer_idx in range(n_blocks)
            ]
        )
        self.output = (
            None
            if d_out is None
            else _named_sequential(
                ('normalization', nn.LayerNorm(d_block)),
                ('activation', nn.ReLU()),
                ('linear', nn.Linear(d_block, d_out)),
            )
        )

    def forward(self, x: Tensor) -> Tensor:

        """Do the forward pass."""

        if x.ndim != 3:
            raise ValueError(
                f'The input must have exactly three dimension, however: {x.ndim=}'
            )
        n_blocks = len(self.blocks)
        for i_block, block in enumerate(self.blocks):
            block = cast(nn.ModuleDict, block)
            x_identity = x
            if 'attention_normalization' in block:
                x = block['attention_normalization'](x)
            x = block['attention'](x[:, :1] if i_block + 1 == n_blocks else x, x)
            x = block['attention_residual_dropout'](x)
            x = x_identity + x
            x_identity = x
            x = block['ffn_normalization'](x)
            x = block['ffn'](x)
            x = block['ffn_residual_dropout'](x)
            x = x_identity + x
            x = block['output'](x)
        x = x[:, 0] # The representation of [CLS]-token.


        if self.output is not None:
            x = self.output(x)
        return x 
        
class FTTransformer(nn.Module):
    """The FT-Transformer model from Section 3.3 in the paper."""

    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        boundaries: Optional[Tensor] = None,
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        # >>> Feature embeddings (Figure 2a in the paper).
        # self.cont_embeddings = (
        # # QuantilePLEEmbedding(boundaries, d_block) if n_cont_features > 0 else None
        # # LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )

        if n_cont_features > 0:
            self.linear_embeddings = LinearEmbeddings(n_cont_features, d_block)
            # self.periodic_embeddings = PeriodicEmbedding(n_cont_features, d_block)
            self.ple_embeddings = QuantilePLEEmbedding(boundaries, d_block)
            self.cont_norm = nn.LayerNorm(d_block)
            # self.cont_embeddings = self.linear_embeddings+self.ple_embeddings
        else: None
        
        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        if x_cont is not None and self.ple_embeddings is not None:
        # if x_cont is not None:
            # 1. Get Global Linear Trend
            lin_tokens = self.linear_embeddings(x_cont)
            # per_tokens = self.periodic_embeddings(x_cont)
            ple_tokens = self.ple_embeddings(x_cont)
            x_embeddings.append(self.cont_norm(lin_tokens + ple_tokens))
            # x_embeddings.append(lin_tokens)

        # Handle Categorical Features
        if x_cat is not None and self.cat_embeddings is not None:
            x_embeddings.append(self.cat_embeddings(x_cat))

        # for argname, argvalue, module in [
        # ('x_cont', x_cont, self.cont_embeddings),
        # ('x_cat', x_cat, self.cat_embeddings),
        # ]:
        # if module is None:
        # if argvalue is not None:
        # raise ValueError(
        # FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
        # f'{argname} must be None'
        # )
        # )
        # else:
        # if argvalue is None:
        # raise ValueError(
        # FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
        # f'{argname} must not be None'
        # )
        # )
        # x_embeddings.append(module(argvalue))
        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 5.2 Train + Evaluate

In [ ]:
BATCH_SIZE=8192+8192
trial_results=[]
global_best_auc=0.7299
global_model_path = str(MODELS_DIR / "ftt_lin_ple_lc.pth")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities,
    boundaries = boundaries_df,
    n_blocks=5,
    d_block=64,
    attention_n_heads=4,
    ffn_d_hidden_multiplier=8/3,
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)
n_epochs = 2 if DEV_MODE else 100
patience = 10
Lr = 0.0002
model = torch.nn.DataParallel(model, device_ids = [0,1]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
# optimizer = Lion(
# model.parameters(),
# lr=Lr,
# betas=(0.9, 0.999), # Lion often works well with beta1=0.9
# weight_decay=1e-5
# )
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 3, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(train_df_proc.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.3,
anneal_strategy="cos"
)

loss_fn = F.binary_cross_entropy_with_logits

def apply_model(batch: Dict[str, Tensor]) -> Tensor:
        return model(batch["X_cont"], batch.get("x_cat")).squeeze(-1)

@torch.no_grad()
def evaluate(part: str) -> float:
    model.eval()
    eval_batch_size = BATCH_SIZE
    y_pred = (
        torch.cat(
            [
                apply_model(batch)
                for batch in delu.iter_batches(data[part], eval_batch_size)
            ]
        )
        .cpu()
        .numpy()
    )
    y_true = data[part]["y"].cpu().numpy()
    y_pred = scipy.special.expit(y_pred)
    auc = roc_auc_score(y_true, y_pred)
    # y_pred = np.round(y_pred)
    # score = sklearn.metrics.accuracy_score(y_true, y_pred>0.3)
    return auc # The higher -- the better.


print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


batch_size = BATCH_SIZE
epoch_size = math.ceil(train_df.shape[0] / batch_size)
timer = delu.tools.Timer()
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            torch.tensor(3.5, device=y_batch.device),
            torch.tensor(1.0, device=y_batch.device)
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    trial_results.append({
        "trial_type": "Q_PLE",
        "d_token": 64,
        "n_blocks": 5,
        "n_heads": 4,
        "ffn_d_hidden": 8/3,
    "epoch_n": epoch,
    "auc_val": val_auc,
    "auc_test": test_auc,
    'best':best,
    'time':timer.__str__(),
    'auc_train':train_auc
      })
    # print(f"(val) {val_score:.4f} (test) {test_score:.4f} [time] {timer}")
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
        break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch, 'train':train_auc}
        if val_auc>global_best_auc:
          torch.save(model.state_dict(), global_model_path)
          global_best_auc=val_auc
    import json
    with open(str(RESULTS_DIR / "ftt_lin_ple_lc_trials.json"), "w") as f:
        json.dump(trial_results, f, indent=4)
    
    # print(trial_results[trial.number])
    # print("\n\nResult:")
    # print(best)
# return best['val']
    # print()

## 6. Variant 2 — MLP-Based Embedding

A small MLP first mixes all numerical features into a context, then each per-feature token uses both its own value and that context.

### 6.1 Model

In [ ]:
import itertools
import math
import typing
import warnings
from collections import OrderedDict
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter

_INTERNAL_ERROR = 'Internal error'
import numpy as np
import torch

def _named_sequential(*modules) -> nn.Sequential:
    return nn.Sequential(OrderedDict(modules))

class ContextualContEmbedding(nn.Module):
    def __init__(self, n_cont_features, d_token):
        super().__init__()
        self.n_cont_features = n_cont_features
        self.d_token = d_token 
        # A small "bottleneck" MLP to mix feature information
        # This allows the model to learn its own "rotation"
        self.mixer = nn.Sequential(
            nn.Linear(n_cont_features, n_cont_features * 4),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(n_cont_features * 4, n_cont_features * d_token)
        )
        # We add a LayerNorm to keep the tokens stable before the Transformer
        self.norm = nn.LayerNorm(d_token)

    def forward(self, x_cont):
        # x_cont shape: [B, n_features]
        B = x_cont.shape[0]
        # 1. Generate mixed tokens
        # Output is a flat vector of [B, n_features * d_token]
        mixed_flat = self.mixer(x_cont)
        # 2. Reshape into tokens for the Transformer backbone
        # Output shape: [B, n_features, d_token]
        tokens = mixed_flat.view(B, self.n_cont_features, self.d_token)
        return self.norm(tokens)

class LinearEmbeddings(nn.Module):
    def __init__(self, n_features: int, d_embedding: int) -> None:
        """
        Args:
            n_features: the number of continous features.
            d_embedding: the embedding size.
        """
        if n_features <= 0:
            raise ValueError(f'n_features must be positive, however: {n_features=}')
        if d_embedding <= 0:
            raise ValueError(f'd_embedding must be positive, however: {d_embedding=}')

        super().__init__()
        self.weight = Parameter(torch.empty(n_features, d_embedding))
        self.bias = Parameter(torch.empty(n_features, d_embedding))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rqsrt = self.weight.shape[1] ** -0.5
        nn.init.uniform_(self.weight, -d_rqsrt, d_rqsrt)
        nn.init.uniform_(self.bias, -d_rqsrt, d_rqsrt)

    def forward(self, x: Tensor) -> Tensor:
        if x.ndim < 2:
            raise ValueError(
                f'The input must have at least two dimensions, however: {x.ndim=}'
            )
        x = x[..., None] * self.weight
        x = x + self.bias[None]
        return x


class CategoricalEmbeddings(nn.Module):
    def __init__(
        self, cardinalities: List[int], d_embedding: int, bias: bool = True
    ) -> None:
        super().__init__()
        if not cardinalities:
            raise ValueError('cardinalities must not be empty')
        if any(x <= 0 for x in cardinalities):
            i, value = next((i, x) for i, x in enumerate(cardinalities) if x <= 0)
            raise ValueError(
                'cardinalities must contain only positive values,'
                f' however: cardinalities[{i}]={value}'
            )
        if d_embedding <= 0:
            raise ValueError(f'd_embedding must be positive, however: {d_embedding=}')

        self.embeddings = nn.ModuleList(
            [nn.Embedding(x, d_embedding) for x in cardinalities]
        )
        self.bias = (
            Parameter(torch.empty(len(cardinalities), d_embedding)) if bias else None
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rsqrt = self.embeddings[0].embedding_dim ** -0.5
        for m in self.embeddings:
            nn.init.uniform_(m.weight, -d_rsqrt, d_rsqrt)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -d_rsqrt, d_rsqrt)

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass."""
        if x.ndim < 2:
            raise ValueError(
                f'The input must have at least two dimensions, however: {x.ndim=}'
            )
        n_features = len(self.embeddings)
        if x.shape[-1] != n_features:
            raise ValueError(
                'The last input dimension (the number of categorical features) must be'
                ' equal to the number of cardinalities passed to the constructor.'
                f' However: {x.shape[-1]=}, len(cardinalities)={n_features}'
            )

        x = torch.stack(
            [self.embeddings[i](x[..., i]) for i in range(n_features)], dim=-2
        )
        if self.bias is not None:
            x = x + self.bias
        return x


_LINFORMER_KV_COMPRESSION_SHARING = Literal['headwise', 'key-value']


class MultiheadAttention(nn.Module):
    def __init__(
        self,
        *,
        d_embedding: int,
        n_heads: int,
        dropout: float,
        # Linformer arguments.
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ) -> None:
        if n_heads < 1:
            raise ValueError(f'n_heads must be positive, however: {n_heads=}')
        if d_embedding % n_heads:
            raise ValueError(
                'd_embedding must be a multiple of n_heads,'
                f' however: {d_embedding=}, {n_heads=}'
            )

        super().__init__()
        self.W_q = nn.Linear(d_embedding, d_embedding)
        self.W_k = nn.Linear(d_embedding, d_embedding)
        self.W_v = nn.Linear(d_embedding, d_embedding)
        self.W_out = nn.Linear(d_embedding, d_embedding) if n_heads > 1 else None
        self.dropout = nn.Dropout(dropout) if dropout else None
        self._n_heads = n_heads

        if linformer_kv_compression_ratio is not None:
            if n_tokens is None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is not None,'
                    ' then n_tokens also must not be None'
                )
            if linformer_kv_compression_sharing not in typing.get_args(
                _LINFORMER_KV_COMPRESSION_SHARING
            ):
                raise ValueError(
                    'Valid values of linformer_kv_compression_sharing include:'
                    f' {typing.get_args(_LINFORMER_KV_COMPRESSION_SHARING)},'
                    f' however: {linformer_kv_compression_sharing=}'
                )
            if (
                linformer_kv_compression_ratio <= 0.0
                or linformer_kv_compression_ratio >= 1.0
            ):
                raise ValueError(
                    'linformer_kv_compression_ratio must be from the open interval'
                    f' (0.0, 1.0), however: {linformer_kv_compression_ratio=}'
                )

            def make_linformer_kv_compression():
                return nn.Linear(
                    n_tokens,
                    max(int(n_tokens * linformer_kv_compression_ratio), 1),
                    bias=False,
                )

            self.key_compression = make_linformer_kv_compression()
            self.value_compression = (
                make_linformer_kv_compression()
                if linformer_kv_compression_sharing == 'headwise'
                else None
            )
        else:
            if n_tokens is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then n_tokens also must be None'
                )
            if linformer_kv_compression_sharing is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then linformer_kv_compression_sharing also must be None'
                )
            self.key_compression = None
            self.value_compression = None

        for m in [self.W_q, self.W_k, self.W_v]:
            nn.init.zeros_(m.bias)
        if self.W_out is not None:
            nn.init.zeros_(self.W_out.bias)

    def _reshape(self, x: Tensor) -> Tensor:
        batch_size, n_tokens, d = x.shape
        d_head = d // self._n_heads
        return (
            x.reshape(batch_size, n_tokens, self._n_heads, d_head)
            .transpose(1, 2)
            .reshape(batch_size * self._n_heads, n_tokens, d_head)
        )

    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        """Do the forward pass."""
        q, k, v = self.W_q(x_q), self.W_k(x_kv), self.W_v(x_kv)
        if self.key_compression is not None:
            k = self.key_compression(k.transpose(1, 2)).transpose(1, 2)
            v = (
                self.key_compression
                if self.value_compression is None
                else self.value_compression
            )(v.transpose(1, 2)).transpose(1, 2)

        batch_size = len(q)
        d_head_key = k.shape[-1] // self._n_heads
        d_head_value = v.shape[-1] // self._n_heads
        n_q_tokens = q.shape[1]

        q = self._reshape(q)
        k = self._reshape(k)
        attention_logits = q @ k.transpose(1, 2) / math.sqrt(d_head_key)
        attention_probs = F.softmax(attention_logits, dim=-1)
        if self.dropout is not None:
            attention_probs = self.dropout(attention_probs)
        x = attention_probs @ self._reshape(v)
        x = (
            x.reshape(batch_size, self._n_heads, n_q_tokens, d_head_value)
            .transpose(1, 2)
            .reshape(batch_size, n_q_tokens, self._n_heads * d_head_value)
        )
        if self.W_out is not None:
            x = self.W_out(x)
        return x

class _ReGLU(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        if x.shape[-1] % 2:
            raise ValueError(
                'For the ReGLU activation, the last input dimension'
                f' must be a multiple of 2, however: {x.shape[-1]=}'
            )
        a, b = x.chunk(2, dim=-1)
        return a * F.relu(b)


_TransformerFFNActivation = Literal['ReLU', 'ReGLU']

class _CLSEmbedding(nn.Module):
    def __init__(self, d_embedding: int) -> None:
        super().__init__()
        self.weight = Parameter(torch.empty(d_embedding))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rsqrt = self.weight.shape[-1] ** -0.5
        nn.init.uniform_(self.weight, -d_rsqrt, d_rsqrt)

    def forward(self, batch_dims: Tuple[int]) -> Tensor:
        if not batch_dims:
            raise ValueError('The input must be non-empty')

        return self.weight.expand(*batch_dims, 1, -1)

class FTTransformerBackbone(nn.Module):
    def __init__(
        self,
        *,
        d_out: Optional[int],
        n_blocks: int,
        d_block: int,
        attention_n_heads: int,
        attention_dropout: float,
        ffn_d_hidden: Optional[int] = None,
        ffn_d_hidden_multiplier: Optional[float],
        ffn_dropout: float,
        # NOTE[DIFF]
        # In the paper, FT-Transformer uses the ReGLU activation.
        # Here, to illustrate the difference, ReLU activation is also supported
        # (in particular, see the docstring).
        ffn_activation: _TransformerFFNActivation = 'ReGLU',
        residual_dropout: float,
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ):
        if ffn_activation not in typing.get_args(_TransformerFFNActivation):
            raise ValueError(
                'ffn_activation must be one of'
                f' {typing.get_args(_TransformerFFNActivation)}.'
                f' However: {ffn_activation=}'
            )
        if ffn_d_hidden is None:
            if ffn_d_hidden_multiplier is None:
                raise ValueError(
                    'If ffn_d_hidden is None,'
                    ' then ffn_d_hidden_multiplier must not be None'
                )
            ffn_d_hidden = int(d_block * cast(float, ffn_d_hidden_multiplier))
        else:
            if ffn_d_hidden_multiplier is not None:
                raise ValueError(
                    'If ffn_d_hidden is not None,'
                    ' then ffn_d_hidden_multiplier must be None'
                )
        super().__init__()
        ffn_use_reglu = ffn_activation == 'ReGLU'
        self.blocks = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        # >>> attention
                        'attention': MultiheadAttention(
                            d_embedding=d_block,
                            n_heads=attention_n_heads,
                            dropout=attention_dropout,
                            n_tokens=n_tokens,
                            linformer_kv_compression_ratio=linformer_kv_compression_ratio,
                            linformer_kv_compression_sharing=linformer_kv_compression_sharing,
                        ),
                        'attention_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> feed-forward
                        'ffn_normalization': nn.LayerNorm(d_block),
                        'ffn': _named_sequential(
                            (
                                'linear1',
                                # ReGLU divides dimension by 2,
                                # so multiplying by 2 to compensate for this.
                                nn.Linear(
                                    d_block, ffn_d_hidden * (2 if ffn_use_reglu else 1)
                                ),
                            ),
                            ('activation', _ReGLU() if ffn_use_reglu else nn.ReLU()),
                            ('dropout', nn.Dropout(ffn_dropout)),
                            ('linear2', nn.Linear(ffn_d_hidden, d_block)),
                        ),
                        'ffn_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> output (for hook-based introspection)
                        'output': nn.Identity(),
                        # >>> the very first normalization
                        **(
                            {}
                            if layer_idx == 0
                            else {'attention_normalization': nn.LayerNorm(d_block)}
                        ),
                    }
                )
                for layer_idx in range(n_blocks)
            ]
        )
        self.output = (
            None
            if d_out is None
            else _named_sequential(
                ('normalization', nn.LayerNorm(d_block)),
                ('activation', nn.ReLU()),
                ('linear', nn.Linear(d_block, d_out)),
            )
        )

    def forward(self, x: Tensor) -> Tensor:

        """Do the forward pass."""

        if x.ndim != 3:
            raise ValueError(
                f'The input must have exactly three dimension, however: {x.ndim=}'
            )
        n_blocks = len(self.blocks)
        for i_block, block in enumerate(self.blocks):
            block = cast(nn.ModuleDict, block)
            x_identity = x
            if 'attention_normalization' in block:
                x = block['attention_normalization'](x)
            x = block['attention'](x[:, :1] if i_block + 1 == n_blocks else x, x)
            x = block['attention_residual_dropout'](x)
            x = x_identity + x
            x_identity = x
            x = block['ffn_normalization'](x)
            x = block['ffn'](x)
            x = block['ffn_residual_dropout'](x)
            x = x_identity + x
            x = block['output'](x)
        x = x[:, 0] # The representation of [CLS]-token.


        if self.output is not None:
            x = self.output(x)
        return x 
        
class FTTransformer(nn.Module):
    """The FT-Transformer model from Section 3.3 in the paper."""

    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        boundaries: Optional[Tensor] = None,
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        #>>> Feature embeddings (Figure 2a in the paper)
        if n_cont_features > 0:
            self.cont_embeddings = ContextualContEmbedding(n_cont_features, d_block)
        
        # self.cont_embeddings = (
        # LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )
        else: None
        
        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        for argname, argvalue, module in [
            ('x_cont', x_cont, self.cont_embeddings),
            ('x_cat', x_cat, self.cat_embeddings),
        ]:
            if module is None:
                if argvalue is not None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must be None'
                        )
                    )
            else:
                if argvalue is None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must not be None'
                        )
                    )
                x_embeddings.append(module(argvalue))
        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 6.2 Train + Evaluate

In [ ]:
BATCH_SIZE=8192+8192
trial_results=[]
global_best_auc=0.7299
global_model_path = str(MODELS_DIR / "ftt_mlp_embed_lc.pth")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities,
    boundaries = boundaries_df,
    n_blocks=5,
    d_block=64,
    attention_n_heads=4,
    ffn_d_hidden_multiplier=8/3,
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)
n_epochs = 2 if DEV_MODE else 100
patience = 10
Lr = 0.0002
model = torch.nn.DataParallel(model, device_ids = [0,1]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
# optimizer = Lion(
# model.parameters(),
# lr=Lr,
# betas=(0.9, 0.999), # Lion often works well with beta1=0.9
# weight_decay=1e-5
# )
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 3, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(train_df_proc.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.3,
anneal_strategy="cos"
)

loss_fn = F.binary_cross_entropy_with_logits

def apply_model(batch: Dict[str, Tensor]) -> Tensor:
        return model(batch["X_cont"], batch.get("x_cat")).squeeze(-1)

@torch.no_grad()
def evaluate(part: str) -> float:
    model.eval()
    eval_batch_size = BATCH_SIZE
    y_pred = (
        torch.cat(
            [
                apply_model(batch)
                for batch in delu.iter_batches(data[part], eval_batch_size)
            ]
        )
        .cpu()
        .numpy()
    )
    y_true = data[part]["y"].cpu().numpy()
    y_pred = scipy.special.expit(y_pred)
    auc = roc_auc_score(y_true, y_pred)
    # y_pred = np.round(y_pred)
    # score = sklearn.metrics.accuracy_score(y_true, y_pred>0.3)
    return auc # The higher -- the better.


print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


batch_size = BATCH_SIZE
epoch_size = math.ceil(train_df.shape[0] / batch_size)
timer = delu.tools.Timer()
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            torch.tensor(3.5, device=y_batch.device),
            torch.tensor(1.0, device=y_batch.device)
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    trial_results.append({
        "trial_type": "Q_PLE",
        "d_token": 64,
        "n_blocks": 5,
        "n_heads": 4,
        "ffn_d_hidden": 8/3,
    "epoch_n": epoch,
    "auc_val": val_auc,
    "auc_test": test_auc,
    'best':best,
    'time':timer.__str__(),
    'auc_train':train_auc
      })
    # print(f"(val) {val_score:.4f} (test) {test_score:.4f} [time] {timer}")
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
        break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch, 'train':train_auc}
        if val_auc>global_best_auc:
          torch.save(model.state_dict(), global_model_path)
          global_best_auc=val_auc
    import json
    with open(str(RESULTS_DIR / "ftt_mlp_embed_lc_trials.json"), "w") as f:
        json.dump(trial_results, f, indent=4)
    
    # print(trial_results[trial.number])
    # print("\n\nResult:")
    # print(best)
# return best['val']
    # print()

## 7. Variant 3 — Cross-SwiGLU Embedding

Similar in spirit to the MLP variant but uses a SwiGLU gate over the cross-feature context, breaking the per-feature independence of vanilla FTT in a different way.

### 7.1 Model

In [ ]:
import itertools
import math
import typing
import warnings
from collections import OrderedDict
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter

_INTERNAL_ERROR = 'Internal error'
import numpy as np
import torch

def _named_sequential(*modules) -> nn.Sequential:
    return nn.Sequential(OrderedDict(modules))

class CrossFeatureSwiGLUTokenizer(nn.Module):
    """
    Cross-feature SwiGLU tokenizer for FTT numerical embeddings.
    
    Breaks the per-feature independence assumption of vanilla FTT by:
    1. Computing a cross-feature context via SwiGLU over all features
    2. Creating each token using both its own value AND the cross-feature context
    
    Drop-in replacement for FTT's cont_embeddings.
    
    Shape:
        Input: (*, n_features)
        Output: (*, n_features, d_block)
    """
    def __init__(self, n_features: int, d_block: int) -> None:
        super().__init__()
        self.n_features = n_features
        self.d_block = d_block
        
        # ── Stage 1: Cross-feature context via SwiGLU ──
        # Maps all features → shared context vector (B, n_features)
        self.context_gate = nn.Linear(n_features, n_features)
        self.context_value = nn.Linear(n_features, n_features)
        
        # ── Stage 2: Per-feature token projection with context ──
        # Each token sees: own scalar (1) + full context (n_features) = n_features + 1
        self.token_gate = nn.Linear(n_features + 1, d_block)
        self.token_value = nn.Linear(n_features + 1, d_block)
        
        self.silu = nn.SiLU()
        self.norm = nn.LayerNorm(d_block)
        
        self._init_weights()
    
    def _init_weights(self) -> None:
        for layer in [
            self.context_gate, self.context_value,
            self.token_gate, self.token_value
        ]:
            nn.init.kaiming_uniform_(layer.weight, a=math.sqrt(5))
            nn.init.zeros_(layer.bias)
    
    def get_output_shape(self) -> torch.Size:
        return torch.Size([self.n_features, self.d_block])
    
    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: (*, n_features) — raw standardized numerical features
        Returns:
            (*, n_features, d_block) — token embeddings
        """
        # ── Stage 1: Cross-feature context ──
        # SwiGLU over all features — each position attends to all others
        context = self.silu(self.context_gate(x)) * self.context_value(x)
        # context: (*, n_features)

        # ── Stage 2: Vectorized per-feature token creation ──
        # Expand context for all n_features token positions
        # Each token i gets: [context (n_features) | own scalar x_i (1)]
        
        # context repeated for each feature position: (*, n_features, n_features)
        context_expanded = context.unsqueeze(-2).expand(
            *context.shape[:-1], self.n_features, self.n_features
        )
        
        # own scalar for each position: (*, n_features, 1)
        x_self = x.unsqueeze(-1)
        
        # concatenate along last dim: (*, n_features, n_features + 1)
        xi_with_context = torch.cat([context_expanded, x_self], dim=-1)
        
        # SwiGLU token projection
        gate = self.silu(self.token_gate(xi_with_context)) # (*, n_features, d_block)
        value = self.token_value(xi_with_context) # (*, n_features, d_block)
        
        out = gate * value # (*, n_features, d_block)
        return self.norm(out)

class LinearEmbeddings(nn.Module):
    def __init__(self, n_features: int, d_embedding: int) -> None:
        """
        Args:
            n_features: the number of continous features.
            d_embedding: the embedding size.
        """
        if n_features <= 0:
            raise ValueError(f'n_features must be positive, however: {n_features=}')
        if d_embedding <= 0:
            raise ValueError(f'd_embedding must be positive, however: {d_embedding=}')

        super().__init__()
        self.weight = Parameter(torch.empty(n_features, d_embedding))
        self.bias = Parameter(torch.empty(n_features, d_embedding))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rqsrt = self.weight.shape[1] ** -0.5
        nn.init.uniform_(self.weight, -d_rqsrt, d_rqsrt)
        nn.init.uniform_(self.bias, -d_rqsrt, d_rqsrt)

    def forward(self, x: Tensor) -> Tensor:
        if x.ndim < 2:
            raise ValueError(
                f'The input must have at least two dimensions, however: {x.ndim=}'
            )
        x = x[..., None] * self.weight
        x = x + self.bias[None]
        return x


class CategoricalEmbeddings(nn.Module):
    def __init__(
        self, cardinalities: List[int], d_embedding: int, bias: bool = True
    ) -> None:
        super().__init__()
        if not cardinalities:
            raise ValueError('cardinalities must not be empty')
        if any(x <= 0 for x in cardinalities):
            i, value = next((i, x) for i, x in enumerate(cardinalities) if x <= 0)
            raise ValueError(
                'cardinalities must contain only positive values,'
                f' however: cardinalities[{i}]={value}'
            )
        if d_embedding <= 0:
            raise ValueError(f'd_embedding must be positive, however: {d_embedding=}')

        self.embeddings = nn.ModuleList(
            [nn.Embedding(x, d_embedding) for x in cardinalities]
        )
        self.bias = (
            Parameter(torch.empty(len(cardinalities), d_embedding)) if bias else None
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rsqrt = self.embeddings[0].embedding_dim ** -0.5
        for m in self.embeddings:
            nn.init.uniform_(m.weight, -d_rsqrt, d_rsqrt)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -d_rsqrt, d_rsqrt)

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass."""
        if x.ndim < 2:
            raise ValueError(
                f'The input must have at least two dimensions, however: {x.ndim=}'
            )
        n_features = len(self.embeddings)
        if x.shape[-1] != n_features:
            raise ValueError(
                'The last input dimension (the number of categorical features) must be'
                ' equal to the number of cardinalities passed to the constructor.'
                f' However: {x.shape[-1]=}, len(cardinalities)={n_features}'
            )

        x = torch.stack(
            [self.embeddings[i](x[..., i]) for i in range(n_features)], dim=-2
        )
        if self.bias is not None:
            x = x + self.bias
        return x


_LINFORMER_KV_COMPRESSION_SHARING = Literal['headwise', 'key-value']


class MultiheadAttention(nn.Module):
    def __init__(
        self,
        *,
        d_embedding: int,
        n_heads: int,
        dropout: float,
        # Linformer arguments.
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ) -> None:
        if n_heads < 1:
            raise ValueError(f'n_heads must be positive, however: {n_heads=}')
        if d_embedding % n_heads:
            raise ValueError(
                'd_embedding must be a multiple of n_heads,'
                f' however: {d_embedding=}, {n_heads=}'
            )

        super().__init__()
        self.W_q = nn.Linear(d_embedding, d_embedding)
        self.W_k = nn.Linear(d_embedding, d_embedding)
        self.W_v = nn.Linear(d_embedding, d_embedding)
        self.W_out = nn.Linear(d_embedding, d_embedding) if n_heads > 1 else None
        self.dropout = nn.Dropout(dropout) if dropout else None
        self._n_heads = n_heads

        if linformer_kv_compression_ratio is not None:
            if n_tokens is None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is not None,'
                    ' then n_tokens also must not be None'
                )
            if linformer_kv_compression_sharing not in typing.get_args(
                _LINFORMER_KV_COMPRESSION_SHARING
            ):
                raise ValueError(
                    'Valid values of linformer_kv_compression_sharing include:'
                    f' {typing.get_args(_LINFORMER_KV_COMPRESSION_SHARING)},'
                    f' however: {linformer_kv_compression_sharing=}'
                )
            if (
                linformer_kv_compression_ratio <= 0.0
                or linformer_kv_compression_ratio >= 1.0
            ):
                raise ValueError(
                    'linformer_kv_compression_ratio must be from the open interval'
                    f' (0.0, 1.0), however: {linformer_kv_compression_ratio=}'
                )

            def make_linformer_kv_compression():
                return nn.Linear(
                    n_tokens,
                    max(int(n_tokens * linformer_kv_compression_ratio), 1),
                    bias=False,
                )

            self.key_compression = make_linformer_kv_compression()
            self.value_compression = (
                make_linformer_kv_compression()
                if linformer_kv_compression_sharing == 'headwise'
                else None
            )
        else:
            if n_tokens is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then n_tokens also must be None'
                )
            if linformer_kv_compression_sharing is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then linformer_kv_compression_sharing also must be None'
                )
            self.key_compression = None
            self.value_compression = None

        for m in [self.W_q, self.W_k, self.W_v]:
            nn.init.zeros_(m.bias)
        if self.W_out is not None:
            nn.init.zeros_(self.W_out.bias)

    def _reshape(self, x: Tensor) -> Tensor:
        batch_size, n_tokens, d = x.shape
        d_head = d // self._n_heads
        return (
            x.reshape(batch_size, n_tokens, self._n_heads, d_head)
            .transpose(1, 2)
            .reshape(batch_size * self._n_heads, n_tokens, d_head)
        )

    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        """Do the forward pass."""
        q, k, v = self.W_q(x_q), self.W_k(x_kv), self.W_v(x_kv)
        if self.key_compression is not None:
            k = self.key_compression(k.transpose(1, 2)).transpose(1, 2)
            v = (
                self.key_compression
                if self.value_compression is None
                else self.value_compression
            )(v.transpose(1, 2)).transpose(1, 2)

        batch_size = len(q)
        d_head_key = k.shape[-1] // self._n_heads
        d_head_value = v.shape[-1] // self._n_heads
        n_q_tokens = q.shape[1]

        q = self._reshape(q)
        k = self._reshape(k)
        attention_logits = q @ k.transpose(1, 2) / math.sqrt(d_head_key)
        attention_probs = F.softmax(attention_logits, dim=-1)
        if self.dropout is not None:
            attention_probs = self.dropout(attention_probs)
        x = attention_probs @ self._reshape(v)
        x = (
            x.reshape(batch_size, self._n_heads, n_q_tokens, d_head_value)
            .transpose(1, 2)
            .reshape(batch_size, n_q_tokens, self._n_heads * d_head_value)
        )
        if self.W_out is not None:
            x = self.W_out(x)
        return x

class _ReGLU(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        if x.shape[-1] % 2:
            raise ValueError(
                'For the ReGLU activation, the last input dimension'
                f' must be a multiple of 2, however: {x.shape[-1]=}'
            )
        a, b = x.chunk(2, dim=-1)
        return a * F.relu(b)


_TransformerFFNActivation = Literal['ReLU', 'ReGLU']

class _CLSEmbedding(nn.Module):
    def __init__(self, d_embedding: int) -> None:
        super().__init__()
        self.weight = Parameter(torch.empty(d_embedding))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rsqrt = self.weight.shape[-1] ** -0.5
        nn.init.uniform_(self.weight, -d_rsqrt, d_rsqrt)

    def forward(self, batch_dims: Tuple[int]) -> Tensor:
        if not batch_dims:
            raise ValueError('The input must be non-empty')

        return self.weight.expand(*batch_dims, 1, -1)

class FTTransformerBackbone(nn.Module):
    def __init__(
        self,
        *,
        d_out: Optional[int],
        n_blocks: int,
        d_block: int,
        attention_n_heads: int,
        attention_dropout: float,
        ffn_d_hidden: Optional[int] = None,
        ffn_d_hidden_multiplier: Optional[float],
        ffn_dropout: float,
        # NOTE[DIFF]
        # In the paper, FT-Transformer uses the ReGLU activation.
        # Here, to illustrate the difference, ReLU activation is also supported
        # (in particular, see the docstring).
        ffn_activation: _TransformerFFNActivation = 'ReGLU',
        residual_dropout: float,
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ):
        if ffn_activation not in typing.get_args(_TransformerFFNActivation):
            raise ValueError(
                'ffn_activation must be one of'
                f' {typing.get_args(_TransformerFFNActivation)}.'
                f' However: {ffn_activation=}'
            )
        if ffn_d_hidden is None:
            if ffn_d_hidden_multiplier is None:
                raise ValueError(
                    'If ffn_d_hidden is None,'
                    ' then ffn_d_hidden_multiplier must not be None'
                )
            ffn_d_hidden = int(d_block * cast(float, ffn_d_hidden_multiplier))
        else:
            if ffn_d_hidden_multiplier is not None:
                raise ValueError(
                    'If ffn_d_hidden is not None,'
                    ' then ffn_d_hidden_multiplier must be None'
                )
        super().__init__()
        ffn_use_reglu = ffn_activation == 'ReGLU'
        self.blocks = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        # >>> attention
                        'attention': MultiheadAttention(
                            d_embedding=d_block,
                            n_heads=attention_n_heads,
                            dropout=attention_dropout,
                            n_tokens=n_tokens,
                            linformer_kv_compression_ratio=linformer_kv_compression_ratio,
                            linformer_kv_compression_sharing=linformer_kv_compression_sharing,
                        ),
                        'attention_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> feed-forward
                        'ffn_normalization': nn.LayerNorm(d_block),
                        'ffn': _named_sequential(
                            (
                                'linear1',
                                # ReGLU divides dimension by 2,
                                # so multiplying by 2 to compensate for this.
                                nn.Linear(
                                    d_block, ffn_d_hidden * (2 if ffn_use_reglu else 1)
                                ),
                            ),
                            ('activation', _ReGLU() if ffn_use_reglu else nn.ReLU()),
                            ('dropout', nn.Dropout(ffn_dropout)),
                            ('linear2', nn.Linear(ffn_d_hidden, d_block)),
                        ),
                        'ffn_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> output (for hook-based introspection)
                        'output': nn.Identity(),
                        # >>> the very first normalization
                        **(
                            {}
                            if layer_idx == 0
                            else {'attention_normalization': nn.LayerNorm(d_block)}
                        ),
                    }
                )
                for layer_idx in range(n_blocks)
            ]
        )
        self.output = (
            None
            if d_out is None
            else _named_sequential(
                ('normalization', nn.LayerNorm(d_block)),
                ('activation', nn.ReLU()),
                ('linear', nn.Linear(d_block, d_out)),
            )
        )

    def forward(self, x: Tensor) -> Tensor:

        """Do the forward pass."""

        if x.ndim != 3:
            raise ValueError(
                f'The input must have exactly three dimension, however: {x.ndim=}'
            )
        n_blocks = len(self.blocks)
        for i_block, block in enumerate(self.blocks):
            block = cast(nn.ModuleDict, block)
            x_identity = x
            if 'attention_normalization' in block:
                x = block['attention_normalization'](x)
            x = block['attention'](x[:, :1] if i_block + 1 == n_blocks else x, x)
            x = block['attention_residual_dropout'](x)
            x = x_identity + x
            x_identity = x
            x = block['ffn_normalization'](x)
            x = block['ffn'](x)
            x = block['ffn_residual_dropout'](x)
            x = x_identity + x
            x = block['output'](x)
        x = x[:, 0] # The representation of [CLS]-token.


        if self.output is not None:
            x = self.output(x)
        return x 
        
class FTTransformer(nn.Module):
    """The FT-Transformer model from Section 3.3 in the paper."""

    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        boundaries: Optional[Tensor] = None,
        _is_default: bool = False,
        **backbone_kwargs,
    ) -> None:
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        #>>> Feature embeddings (Figure 2a in the paper)
        if n_cont_features > 0:
            self.cont_embeddings = CrossFeatureSwiGLUTokenizer(n_cont_features, d_block)
        
        # self.cont_embeddings = (
        # LinearEmbeddings(n_cont_features, d_block) if n_cont_features > 0 else None
        # )
        else: None
        
        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        for argname, argvalue, module in [
            ('x_cont', x_cont, self.cont_embeddings),
            ('x_cat', x_cat, self.cat_embeddings),
        ]:
            if module is None:
                if argvalue is not None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must be None'
                        )
                    )
            else:
                if argvalue is None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must not be None'
                        )
                    )
                x_embeddings.append(module(argvalue))
        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 7.2 Train + Evaluate

In [ ]:
BATCH_SIZE=8192+8192
trial_results=[]
global_best_auc=0.7299
global_model_path = str(MODELS_DIR / "ftt_cross_swiglu_lc.pth")
model = FTTransformer(
    n_cont_features=d_numerical,
    cat_cardinalities=cat_cardinalities,
    boundaries = boundaries_df,
    n_blocks=5,
    d_block=64,
    attention_n_heads=4,
    ffn_d_hidden_multiplier=8/3,
    attention_dropout=0.05,
    ffn_dropout=0.05,
    residual_dropout=0.05,
    d_out=1 # output a single logit (binary)
).to(device)
n_epochs = 2 if DEV_MODE else 100
patience = 10
Lr = 0.0002
model = torch.nn.DataParallel(model, device_ids = [0,1]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
# optimizer = Lion(
# model.parameters(),
# lr=Lr,
# betas=(0.9, 0.999), # Lion often works well with beta1=0.9
# weight_decay=1e-5
# )
scheduler = torch.optim.lr_scheduler.OneCycleLR(
optimizer,
max_lr=Lr * 3, # Recommended: 3–10× base LR
steps_per_epoch=math.ceil(train_df_proc.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
epochs=n_epochs,
pct_start=0.3,
anneal_strategy="cos"
)

loss_fn = F.binary_cross_entropy_with_logits

def apply_model(batch: Dict[str, Tensor]) -> Tensor:
        return model(batch["X_cont"], batch.get("x_cat")).squeeze(-1)

@torch.no_grad()
def evaluate(part: str) -> float:
    model.eval()
    eval_batch_size = BATCH_SIZE
    y_pred = (
        torch.cat(
            [
                apply_model(batch)
                for batch in delu.iter_batches(data[part], eval_batch_size)
            ]
        )
        .cpu()
        .numpy()
    )
    y_true = data[part]["y"].cpu().numpy()
    y_pred = scipy.special.expit(y_pred)
    auc = roc_auc_score(y_true, y_pred)
    # y_pred = np.round(y_pred)
    # score = sklearn.metrics.accuracy_score(y_true, y_pred>0.3)
    return auc # The higher -- the better.


print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


batch_size = BATCH_SIZE
epoch_size = math.ceil(train_df.shape[0] / batch_size)
timer = delu.tools.Timer()
early_stopping = delu.tools.EarlyStopping(patience, mode="max")
best = {
    "val": -math.inf,
    "test": -math.inf,
    "epoch": -1,
}

print(f"Device: {device.type.upper()}")
print("-" * 88 + "\n")
timer.run()
for epoch in range(n_epochs):
    for batch in tqdm(
        delu.iter_batches(data["train"], batch_size, shuffle=True),
        desc=f"Epoch {epoch}",
        total=epoch_size,
    ):
        model.train()
        optimizer.zero_grad()

        y_batch = batch["y"].float()
        weights = torch.where(
            y_batch == 1,
            torch.tensor(3.5, device=y_batch.device),
            torch.tensor(1.0, device=y_batch.device)
        )

        loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
        loss.backward()
        optimizer.step()
        scheduler.step()

    val_auc = evaluate("val")
    test_auc = evaluate("test")
    train_auc = evaluate('train')
    trial_results.append({
        "trial_type": "Q_PLE",
        "d_token": 64,
        "n_blocks": 5,
        "n_heads": 4,
        "ffn_d_hidden": 8/3,
    "epoch_n": epoch,
    "auc_val": val_auc,
    "auc_test": test_auc,
    'best':best,
    'time':timer.__str__(),
    'auc_train':train_auc
      })
    # print(f"(val) {val_score:.4f} (test) {test_score:.4f} [time] {timer}")
    print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

    early_stopping.update(val_auc)
    if early_stopping.should_stop():
        break

    if val_auc > best["val"]:
        print(" New best epoch! ")
        best = {"val": val_auc, "test": test_auc, "epoch": epoch, 'train':train_auc}
        if val_auc>global_best_auc:
          torch.save(model.state_dict(), global_model_path)
          global_best_auc=val_auc
    import json
    with open(str(RESULTS_DIR / "ftt_cross_swiglu_lc_trials.json"), "w") as f:
        json.dump(trial_results, f, indent=4)
    
    # print(trial_results[trial.number])
    # print("\n\nResult:")
    # print(best)
# return best['val']
    # print()

## 8. Variant 4 — Periodic Embedding

Periodic (sin/cos) embeddings as proposed in Gorishniy, Rubachev and Babenko (2022). Useful for representing fine-grained continuous structure that quantile bins might miss.

### 8.1 Model

In [ ]:
# __version__ = '0.0.2'
import itertools
import math
import typing
import warnings
from collections import OrderedDict
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter

_INTERNAL_ERROR = 'Internal error'



def _named_sequential(*modules) -> nn.Sequential:
    return nn.Sequential(OrderedDict(modules))

def _check_input_shape(x: Tensor, expected_n_features: int) -> None:
    if x.ndim < 1:
        raise ValueError(
            f'The input must have at least one dimension, however: {x.ndim=}'
        )
    if x.shape[-1] != expected_n_features:
        raise ValueError(
            'The last dimension of the input was expected to be'
            f' {expected_n_features}, however, {x.shape[-1]=}'
        )


class _Periodic(nn.Module):
    """
    NOTE: THIS MODULE SHOULD NOT BE USED DIRECTLY.

    Technically, this is a linear embedding without bias followed by
    the periodic activations. The scale of the initialization
    (defined by the `sigma` argument) plays an important role.
    """

    def __init__(self, n_features: int, k: int, sigma: float) -> None:
        if sigma <= 0.0:
            raise ValueError(f'sigma must be positive, however: {sigma=}')

        super().__init__()
        self._sigma = sigma
        self.weight = Parameter(torch.empty(n_features, k))
        self.reset_parameters()

    def reset_parameters(self):
        """Reset the parameters."""
        # NOTE[DIFF]
        # Here, extreme values (~0.3% probability) are explicitly avoided just in case.
        # In the paper, there was no protection from extreme values.
        bound = self._sigma * 3
        nn.init.trunc_normal_(self.weight, 0.0, self._sigma, a=-bound, b=bound)

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass."""
        _check_input_shape(x, self.weight.shape[0])
        x = 2 * math.pi * self.weight * x[..., None]
        x = torch.cat([torch.cos(x), torch.sin(x)], -1)
        return x

class _NLinear(nn.Module):
    """N *separate* linear layers for N feature embeddings.

    In other words,
    each feature embedding is transformed by its own dedicated linear layer.
    """

    def __init__(
        self, n: int, in_features: int, out_features: int, bias: bool = True
    ) -> None:
        super().__init__()
        self.weight = Parameter(torch.empty(n, in_features, out_features))
        self.bias = Parameter(torch.empty(n, out_features)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        """Reset the parameters."""
        d_in_rsqrt = self.weight.shape[-2] ** -0.5
        nn.init.uniform_(self.weight, -d_in_rsqrt, d_in_rsqrt)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -d_in_rsqrt, d_in_rsqrt)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Do the forward pass."""
        if x.ndim != 3:
            raise ValueError(
                '_NLinear supports only inputs with exactly one batch dimension,'
                ' so `x` must have a shape like (BATCH_SIZE, N_FEATURES, D_EMBEDDING).'
            )
        assert x.shape[-(self.weight.ndim - 1) :] == self.weight.shape[:-1]

        x = x.transpose(0, 1)
        x = x @ self.weight
        x = x.transpose(0, 1)
        if self.bias is not None:
            x = x + self.bias
        return x

class LinearEmbeddings(nn.Module):
      def __init__(
        self,
        n_features: int,
        d_embedding: int = 24,
        *,
        n_frequencies: int = 48,
        frequency_init_scale: float = 0.01,
        activation: bool = True,
        lite: bool=False,
    ) -> None:
        """
        Args:
            n_features: the number of features.
            d_embedding: the embedding size.
            n_frequencies: the number of frequencies for each feature.
                (denoted as "k" in Section 3.3 in the paper).
            frequency_init_scale: the initialization scale for the first linear layer
                (denoted as "sigma" in Section 3.3 in the paper).
                **This is an important hyperparameter**, see README for details.
            activation: if `False`, the ReLU activation is not applied.
                Must be `True` if ``lite=True``.
            lite: if True, the outer linear layer is shared between all features.
                See README for details.
        """
        super().__init__()
        self.periodic = _Periodic(n_features, n_frequencies, frequency_init_scale)
        self.linear: Union[nn.Linear, _NLinear]
        if lite:
            # NOTE[DIFF]
            # The lite variation was introduced in a different paper
            # (about the TabR model).
            if not activation:
                raise ValueError('lite=True is allowed only when activation=True')
            self.linear = nn.Linear(2 * n_frequencies, d_embedding)
        else:
            self.linear = _NLinear(n_features, 2 * n_frequencies, d_embedding)
        self.activation = nn.ReLU() if activation else None

      def get_output_shape(self) -> torch.Size:
          """Get the output shape without the batch dimensions."""
          n_features = self.periodic.weight.shape[0]
          d_embedding = (
              self.linear.weight.shape[0]
              if isinstance(self.linear, nn.Linear)
              else self.linear.weight.shape[-1]
          )
          return torch.Size((n_features, d_embedding))

      def forward(self, x: Tensor) -> Tensor:
          """Do the forward pass."""
          x = self.periodic(x)
          x = self.linear(x)
          if self.activation is not None:
              x = self.activation(x)
          return x

class CategoricalEmbeddings(nn.Module):
    """Embeddings for categorical features.

    **Examples**

    >>> cardinalities = [3, 10]
    >>> x = torch.tensor([
    ... [0, 5],
    ... [1, 7],
    ... [0, 2],
    ... [2, 4]
    ... ])
    >>> x.shape # (batch_size, n_cat_features)
    torch.Size([4, 2])
    >>> m = CategoricalEmbeddings(cardinalities, d_embedding=5)
    >>> m(x).shape # (batch_size, n_cat_features, d_embedding)
    torch.Size([4, 2, 5])
    """

    def __init__(
        self, cardinalities: List[int], d_embedding: int, bias: bool = True
    ) -> None:
        """
        Args:
            cardinalities: the number of distinct values for each feature.
            d_embedding: the embedding size.
            bias: if `True`, for each feature, a trainable vector is added to the
                embedding regardless of a feature value. For each feature, a separate
                non-shared bias vector is allocated.
                In the paper, FT-Transformer uses `bias=True`.
        """
        super().__init__()
        if not cardinalities:
            raise ValueError('cardinalities must not be empty')
        if any(x <= 0 for x in cardinalities):
            i, value = next((i, x) for i, x in enumerate(cardinalities) if x <= 0)
            raise ValueError(
                'cardinalities must contain only positive values,'
                f' however: cardinalities[{i}]={value}'
            )
        if d_embedding <= 0:
            raise ValueError(f'd_embedding must be positive, however: {d_embedding=}')

        self.embeddings = nn.ModuleList(
            [nn.Embedding(x, d_embedding) for x in cardinalities]
        )
        self.bias = (
            Parameter(torch.empty(len(cardinalities), d_embedding)) if bias else None
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rsqrt = self.embeddings[0].embedding_dim ** -0.5
        for m in self.embeddings:
            nn.init.uniform_(m.weight, -d_rsqrt, d_rsqrt)
        if self.bias is not None:
            nn.init.uniform_(self.bias, -d_rsqrt, d_rsqrt)

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass."""
        if x.ndim < 2:
            raise ValueError(
                f'The input must have at least two dimensions, however: {x.ndim=}'
            )
        n_features = len(self.embeddings)
        if x.shape[-1] != n_features:
            raise ValueError(
                'The last input dimension (the number of categorical features) must be'
                ' equal to the number of cardinalities passed to the constructor.'
                f' However: {x.shape[-1]=}, len(cardinalities)={n_features}'
            )

        x = torch.stack(
            [self.embeddings[i](x[..., i]) for i in range(n_features)], dim=-2
        )
        if self.bias is not None:
            x = x + self.bias
        return x
 

_LINFORMER_KV_COMPRESSION_SHARING = Literal['headwise', 'key-value']


class MultiheadAttention(nn.Module):
    """Multihead (Self-/Cross-)Attention with an optional linear attention from ["Linformer: Self-Attention with Linear Complexity"](https://arxiv.org/abs/2006.04768).

    **Examples**

    >>> batch_size, n_tokens, d_embedding = 2, 3, 16
    >>> n_heads = 8
    >>> a = torch.randn(batch_size, n_tokens, d_embedding)
    >>> b = torch.randn(batch_size, n_tokens * 2, d_embedding)
    >>> m = MultiheadAttention(
    ... d_embedding=d_embedding, n_heads=n_heads, dropout=0.2
    >>> )
    >>>
    >>> # Self-attention.
    >>> assert m(a, a).shape == a.shape
    >>>
    >>> # Cross-attention.
    >>> assert m(a, b).shape == a.shape
    >>>
    >>> # Linformer attention.
    >>> m = MultiheadAttention(
    ... d_embedding=d_embedding,
    ... n_heads=n_heads,
    ... dropout=0.2,
    ... n_tokens=n_tokens,
    ... linformer_kv_compression_ratio=0.5,
    ... linformer_kv_compression_sharing='headwise',
    >>> )
    >>> assert m(a, a).shape == a.shape
    """ # noqa: E501

    def __init__(
        self,
        *,
        d_embedding: int,
        n_heads: int,
        dropout: float,
        # Linformer arguments.
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ) -> None:
        """
        Args:
            d_embedding: the embedding size for one token.
                Must be a multiple of `n_heads`.
            n_heads: the number of heads. If greater than 1, then the module will have
                an additional output layer (the so called "mixing" layer).
            dropout: the dropout rate for the attention probability map.
            n_tokens: the number of tokens
                (must be provided if `linformer_kv_compression_ratio` is not None)
            linformer_kv_compression_ratio: Linformer-style compression rate.
                Must be within the interval `(0.0, 1.0)`.
            linformer_kv_compression_sharing: Linformer compression sharing policy.
                Must be provided if `linformer_kv_compression_ratio` is not None.
                (non-shared Linformer compression is not supported; the "layerwise"
                sharing policy is not supported).
        """
        if n_heads < 1:
            raise ValueError(f'n_heads must be positive, however: {n_heads=}')
        if d_embedding % n_heads:
            raise ValueError(
                'd_embedding must be a multiple of n_heads,'
                f' however: {d_embedding=}, {n_heads=}'
            )

        super().__init__()
        self.W_q = nn.Linear(d_embedding, d_embedding)
        self.W_k = nn.Linear(d_embedding, d_embedding)
        self.W_v = nn.Linear(d_embedding, d_embedding)
        self.W_out = nn.Linear(d_embedding, d_embedding) if n_heads > 1 else None
        self.dropout = nn.Dropout(dropout) if dropout else None
        self._n_heads = n_heads

        if linformer_kv_compression_ratio is not None:
            if n_tokens is None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is not None,'
                    ' then n_tokens also must not be None'
                )
            if linformer_kv_compression_sharing not in typing.get_args(
                _LINFORMER_KV_COMPRESSION_SHARING
            ):
                raise ValueError(
                    'Valid values of linformer_kv_compression_sharing include:'
                    f' {typing.get_args(_LINFORMER_KV_COMPRESSION_SHARING)},'
                    f' however: {linformer_kv_compression_sharing=}'
                )
            if (
                linformer_kv_compression_ratio <= 0.0
                or linformer_kv_compression_ratio >= 1.0
            ):
                raise ValueError(
                    'linformer_kv_compression_ratio must be from the open interval'
                    f' (0.0, 1.0), however: {linformer_kv_compression_ratio=}'
                )

            def make_linformer_kv_compression():
                return nn.Linear(
                    n_tokens,
                    max(int(n_tokens * linformer_kv_compression_ratio), 1),
                    bias=False,
                )

            self.key_compression = make_linformer_kv_compression()
            self.value_compression = (
                make_linformer_kv_compression()
                if linformer_kv_compression_sharing == 'headwise'
                else None
            )
        else:
            if n_tokens is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then n_tokens also must be None'
                )
            if linformer_kv_compression_sharing is not None:
                raise ValueError(
                    'If linformer_kv_compression_ratio is None,'
                    ' then linformer_kv_compression_sharing also must be None'
                )
            self.key_compression = None
            self.value_compression = None

        for m in [self.W_q, self.W_k, self.W_v]:
            nn.init.zeros_(m.bias)
        if self.W_out is not None:
            nn.init.zeros_(self.W_out.bias)

    def _reshape(self, x: Tensor) -> Tensor:
        batch_size, n_tokens, d = x.shape
        d_head = d // self._n_heads
        return (
            x.reshape(batch_size, n_tokens, self._n_heads, d_head)
            .transpose(1, 2)
            .reshape(batch_size * self._n_heads, n_tokens, d_head)
        )

    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        """Do the forward pass."""
        q, k, v = self.W_q(x_q), self.W_k(x_kv), self.W_v(x_kv)
        if self.key_compression is not None:
            k = self.key_compression(k.transpose(1, 2)).transpose(1, 2)
            v = (
                self.key_compression
                if self.value_compression is None
                else self.value_compression
            )(v.transpose(1, 2)).transpose(1, 2)

        batch_size = len(q)
        d_head_key = k.shape[-1] // self._n_heads
        d_head_value = v.shape[-1] // self._n_heads
        n_q_tokens = q.shape[1]

        q = self._reshape(q)
        k = self._reshape(k)
        attention_logits = q @ k.transpose(1, 2) / math.sqrt(d_head_key)
        attention_probs = F.softmax(attention_logits, dim=-1)
        if self.dropout is not None:
            attention_probs = self.dropout(attention_probs)
        x = attention_probs @ self._reshape(v)
        x = (
            x.reshape(batch_size, self._n_heads, n_q_tokens, d_head_value)
            .transpose(1, 2)
            .reshape(batch_size, n_q_tokens, self._n_heads * d_head_value)
        )
        if self.W_out is not None:
            x = self.W_out(x)
        return x


class _ReGLU(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        if x.shape[-1] % 2:
            raise ValueError(
                'For the ReGLU activation, the last input dimension'
                f' must be a multiple of 2, however: {x.shape[-1]=}'
            )
        a, b = x.chunk(2, dim=-1)
        return a * F.relu(b)


_TransformerFFNActivation = Literal['ReLU', 'ReGLU']


class FTTransformerBackbone(nn.Module):
    """The backbone of FT-Transformer.

    The differences with Transformer from the paper
    ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) are as follows:

    - the so called "PreNorm" variation is used
        (`norm_first=True` in terms of `torch.nn.TransformerEncoderLayer`)
    - the very first normalization is skipped. This is **CRUCIAL** for FT-Transformer
        in the PreNorm configuration.

    **Examples**

    >>> batch_size = 2
    >>> n_tokens = 3
    >>> d_block = 16
    >>> x = torch.randn(batch_size, n_tokens, d_block)
    >>> d_out = 1
    >>> m = FTTransformerBackbone(
    ... d_out=d_out,
    ... n_blocks=2,
    ... d_block=d_block,
    ... attention_n_heads=8,
    ... attention_dropout=0.2,
    ... ffn_d_hidden=None,
    ... ffn_d_hidden_multiplier=2.0,
    ... ffn_dropout=0.1,
    ... residual_dropout=0.0,
    ... )
    >>> m(x).shape
    torch.Size([2, 1])
    """

    def __init__(
        self,
        *,
        d_out: Optional[int],
        n_blocks: int,
        d_block: int,
        attention_n_heads: int,
        attention_dropout: float,
        ffn_d_hidden: Optional[int] = None,
        ffn_d_hidden_multiplier: Optional[float],
        ffn_dropout: float,
        # NOTE[DIFF]
        # In the paper, FT-Transformer uses the ReGLU activation.
        # Here, to illustrate the difference, ReLU activation is also supported
        # (in particular, see the docstring).
        ffn_activation: _TransformerFFNActivation = 'ReGLU',
        residual_dropout: float,
        n_tokens: Optional[int] = None,
        linformer_kv_compression_ratio: Optional[float] = None,
        linformer_kv_compression_sharing: Optional[
            _LINFORMER_KV_COMPRESSION_SHARING
        ] = None,
    ):
        """
        Args:
            d_out: the output size.
            n_blocks: the number of blocks.
            d_block: the block width
                (or, equivalently, the embedding size of each feature).
                Must be a multiple of `attention_n_heads`.
            attention_n_heads: the number of attention heads in `MultiheadAttention`.
            attention_dropout: the dropout rate in `MultiheadAttention`. Usually,
                positive values work better, even if the number of features is low.
            ffn_d_hidden: the hidden representation size after the activation in the
                feed-forward blocks (or, equivalently, the *input* size of the *second*
                linear layer in the feed-forward blocks). If ``ffn_use_reglu``
                is `True`, then the *output* size of the *first* linear layer
                will be set to ``2 * ffn_d_hidden``.
            ffn_d_hidden_multiplier: the alternative way to set `ffn_d_hidden` as
                `int(d_block * ffn_d_hidden_multiplier)`.
            ffn_dropout: the dropout rate for the hidden representation
                in the feed-forward blocks.
            ffn_activation: the activation used in the FFN blocks. To maintain (almost)
                the same number of parameters between different activations:
                <ffn_d_hidden_multiplier for ReGLU> = <2 / 3 * ffn_d_hidden_multiplier for ReLU>
                or
                <ffn_d_hidden_multiplier for ReLU> = <3 / 2 * ffn_d_hidden_multiplier for ReGLU>
            residual_dropout: the dropout rate for all residual branches.
            n_tokens: the argument for `MultiheadAttention`.
            linformer_kv_compression_ratio: the argument for `MultiheadAttention`.
            linformer_kv_compression_sharing: the argument for `MultiheadAttention`.
        """ # noqa: E501
        if ffn_activation not in typing.get_args(_TransformerFFNActivation):
            raise ValueError(
                'ffn_activation must be one of'
                f' {typing.get_args(_TransformerFFNActivation)}.'
                f' However: {ffn_activation=}'
            )
        if ffn_d_hidden is None:
            if ffn_d_hidden_multiplier is None:
                raise ValueError(
                    'If ffn_d_hidden is None,'
                    ' then ffn_d_hidden_multiplier must not be None'
                )
            ffn_d_hidden = int(d_block * cast(float, ffn_d_hidden_multiplier))
        else:
            if ffn_d_hidden_multiplier is not None:
                raise ValueError(
                    'If ffn_d_hidden is not None,'
                    ' then ffn_d_hidden_multiplier must be None'
                )

        super().__init__()
        ffn_use_reglu = ffn_activation == 'ReGLU'
        self.blocks = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        # >>> attention
                        'attention': MultiheadAttention(
                            d_embedding=d_block,
                            n_heads=attention_n_heads,
                            dropout=attention_dropout,
                            n_tokens=n_tokens,
                            linformer_kv_compression_ratio=linformer_kv_compression_ratio,
                            linformer_kv_compression_sharing=linformer_kv_compression_sharing,
                        ),
                        'attention_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> feed-forward
                        'ffn_normalization': nn.LayerNorm(d_block),
                        'ffn': _named_sequential(
                            (
                                'linear1',
                                # ReGLU divides dimension by 2,
                                # so multiplying by 2 to compensate for this.
                                nn.Linear(
                                    d_block, ffn_d_hidden * (2 if ffn_use_reglu else 1)
                                ),
                            ),
                            ('activation', _ReGLU() if ffn_use_reglu else nn.ReLU()),
                            ('dropout', nn.Dropout(ffn_dropout)),
                            ('linear2', nn.Linear(ffn_d_hidden, d_block)),
                        ),
                        'ffn_residual_dropout': nn.Dropout(residual_dropout),
                        # >>> output (for hook-based introspection)
                        'output': nn.Identity(),
                        # >>> the very first normalization
                        **(
                            {}
                            if layer_idx == 0
                            else {'attention_normalization': nn.LayerNorm(d_block)}
                        ),
                    }
                )
                for layer_idx in range(n_blocks)
            ]
        )
        self.output = (
            None
            if d_out is None
            else _named_sequential(
                ('normalization', nn.LayerNorm(d_block)),
                ('activation', nn.ReLU()),
                ('linear', nn.Linear(d_block, d_out)),
            )
        )

    def forward(self, x: Tensor) -> Tensor:
        """Do the forward pass."""
        if x.ndim != 3:
            raise ValueError(
                f'The input must have exactly three dimension, however: {x.ndim=}'
            )

        n_blocks = len(self.blocks)
        for i_block, block in enumerate(self.blocks):
            block = cast(nn.ModuleDict, block)

            x_identity = x
            if 'attention_normalization' in block:
                x = block['attention_normalization'](x)
            x = block['attention'](x[:, :1] if i_block + 1 == n_blocks else x, x)
            x = block['attention_residual_dropout'](x)
            x = x_identity + x

            x_identity = x
            x = block['ffn_normalization'](x)
            x = block['ffn'](x)
            x = block['ffn_residual_dropout'](x)
            x = x_identity + x

            x = block['output'](x)

        x = x[:, 0] # The representation of [CLS]-token.

        if self.output is not None:
            x = self.output(x)
        return x


class _CLSEmbedding(nn.Module):
    def __init__(self, d_embedding: int) -> None:
        super().__init__()
        self.weight = Parameter(torch.empty(d_embedding))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        d_rsqrt = self.weight.shape[-1] ** -0.5
        nn.init.uniform_(self.weight, -d_rsqrt, d_rsqrt)

    def forward(self, batch_dims: Tuple[int]) -> Tensor:
        if not batch_dims:
            raise ValueError('The input must be non-empty')

        return self.weight.expand(*batch_dims, 1, -1)


class FTTransformer(nn.Module):
    """The FT-Transformer model from Section 3.3 in the paper."""

    def __init__(
        self,
        *,
        n_cont_features: int,
        cat_cardinalities: List[int],
        _is_default: bool = False,
        n_frequencies: int = 48,
        **backbone_kwargs,
    ) -> None:
        """
        Args:
            n_cont_features: the number of continuous features.
            cat_cardinalities: the cardinalities of categorical features.
                Pass en empty list if there are no categorical features.
            _is_default: this is a technical argument, don't set it manually.
            backbone_kwargs: the keyword arguments for the `FTTransformerBackbone`.
        """
        if n_cont_features < 0:
            raise ValueError(
                f'n_cont_features must be non-negative, however: {n_cont_features=}'
            )
        if n_cont_features == 0 and not cat_cardinalities:
            raise ValueError(
                'At least one type of features must be presented, however:'
                f' {n_cont_features=}, {cat_cardinalities=}'
            )
        if 'n_tokens' in backbone_kwargs:
            raise ValueError(
                'backbone_kwargs must not contain key "n_tokens"'
                ' (the number of tokens will be inferred automatically)'
            )

        super().__init__()
        d_block: int = backbone_kwargs['d_block']
        self.cls_embedding = _CLSEmbedding(d_block)

        # >>> Feature embeddings (Figure 2a in the paper).
        self.cont_embeddings = (
            LinearEmbeddings(n_cont_features, d_block,n_frequencies=n_frequencies) if n_cont_features > 0 else None
        )
        self.cat_embeddings = (
            CategoricalEmbeddings(cat_cardinalities, d_block, True)
            if cat_cardinalities
            else None
        )
        # <<<

        self.backbone = FTTransformerBackbone(
            **backbone_kwargs,
            n_tokens=(
                None
                if backbone_kwargs.get('linformer_kv_compression_ratio') is None
                else 1 + n_cont_features + len(cat_cardinalities)
            ),
        )
        self._is_default = _is_default

    @classmethod
    def get_default_kwargs(cls, n_blocks: int = 3) -> Dict[str, Any]:
        """Get the default hyperparameters.

        Args:
            n_blocks: the number of blocks. The supported values are: 1, 2, 3, 4, 5, 6.
        Returns:
            the default keyword arguments for the constructor.
        """
        if n_blocks < 0 or n_blocks > 6:
            raise ValueError(
                'Default configurations are available'
                ' only for the following values of n_blocks: 1, 2, 3, 4, 5, 6.'
                f' However, {n_blocks=}'
            )
        return {
            'n_blocks': n_blocks,
            'd_block': [96, 128, 192, 256, 320, 384][n_blocks - 1],
            'attention_n_heads': 8,
            'attention_dropout': [0.1, 0.15, 0.2, 0.25, 0.3, 0.35][n_blocks - 1],
            'ffn_d_hidden': None,
            # "4 / 3" for ReGLU leads to almost the same number of parameters
            # as "2.0" for ReLU.
            'ffn_d_hidden_multiplier': 4 / 3,
            'ffn_dropout': [0.0, 0.05, 0.1, 0.15, 0.2, 0.25][n_blocks - 1],
            'residual_dropout': 0.0,
            '_is_default': True,
        }

    def make_parameter_groups(self) -> List[Dict[str, Any]]:
        """Make parameter groups for optimizers.

        The difference with calling this method instead of
        `.parameters()` is that this method always sets `weight_decay=0.0`
        for some of the parameters.

        Returns:
            the parameter groups that can be passed to PyTorch optimizers.
        """

        def get_parameters(m: Optional[nn.Module]) -> Iterable[Parameter]:
            return () if m is None else m.parameters()

        zero_wd_group: Dict[str, Any] = {
            'params': set(
                itertools.chain(
                    get_parameters(self.cls_embedding),
                    get_parameters(self.cont_embeddings),
                    get_parameters(self.cat_embeddings),
                    itertools.chain.from_iterable(
                        m.parameters()
                        for block in self.backbone.blocks
                        for name, m in block.named_children()
                        if name.endswith('_normalization')
                    ),
                    (
                        p
                        for name, p in self.named_parameters()
                        if name.endswith('.bias')
                    ),
                )
            ),
            'weight_decay': 0.0,
        }
        main_group: Dict[str, Any] = {
            'params': [p for p in self.parameters() if p not in zero_wd_group['params']]
        }
        zero_wd_group['params'] = list(zero_wd_group['params'])
        return [main_group, zero_wd_group]

    def make_default_optimizer(self) -> torch.optim.AdamW:
        """Create the "default" `torch.nn.AdamW` suitable for the *default* FT-Transformer.

        Returns:
            the optimizer.
        """ # noqa: E501
        if not self._is_default:
            warnings.warn(
                'The default opimizer is supposed to be used in a combination'
                ' with the default FT-Transformer.'
            )
        return torch.optim.AdamW(
            self.make_parameter_groups(), lr=1e-4, weight_decay=1e-5
        )

    _FORWARD_BAD_ARGS_MESSAGE = (
        'Based on the arguments passed to the constructor of FTTransformer, {}'
    )

    def forward(self, x_cont: Optional[Tensor], x_cat: Optional[Tensor]) -> Tensor:
        """Do the forward pass."""
        x_any = x_cat if x_cont is None else x_cont
        if x_any is None:
            raise ValueError('At least one of x_cont and x_cat must be provided.')

        x_embeddings: List[Tensor] = []
        if self.cls_embedding is not None:
            x_embeddings.append(self.cls_embedding(x_any.shape[:-1]))

        for argname, argvalue, module in [
            ('x_cont', x_cont, self.cont_embeddings),
            ('x_cat', x_cat, self.cat_embeddings),
        ]:
            if module is None:
                if argvalue is not None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must be None'
                        )
                    )
            else:
                if argvalue is None:
                    raise ValueError(
                        FTTransformer._FORWARD_BAD_ARGS_MESSAGE.format(
                            f'{argname} must not be None'
                        )
                    )
                x_embeddings.append(module(argvalue))
        assert x_embeddings, _INTERNAL_ERROR
        x = torch.cat(x_embeddings, dim=1)
        x = self.backbone(x)
        return x

### 8.2 Train + Evaluate

In [ ]:
import json
# print(data['test']['X_cont'].device)
# data = {
# part: {k: torch.as_tensor(v.to_numpy()).to(device) for k, v in data_numpy[part].items()}
# for part in data_numpy
# }

trial_results = [] # stores AUC + hyperparams

d_out=1
BATCH_SIZE = 8192
# from rtdl_revisiting_models import FTTransformer
import math
def optim(trial):
    # -----------------------
    # Search Space - shallow nets
    # -----------------------
    d_Block = trial.suggest_categorical("d_token",[192, 256]) #
    n_Blocks = trial.suggest_int("n_blocks", 2, 6)
    Attention_n_heads = trial.suggest_categorical("attention_n_heads", [2, 4, 8]) #
    FFN_d_hidden = trial.suggest_categorical("ffn_d_hidden", [4/3, 2/3, 6/3])
    Attention_dropout = trial.suggest_float("attention_dropout", 0.0, 0.1)
    n_frequencies = trial.suggest_int("n_frequencies", 20, 80)
    FFN_dropout = trial.suggest_float("ffn_dropout", 0.0, 0.1)
    Residual_droupout = trial.suggest_float('residual_droupout',0,0.1)
    Lr = trial.suggest_float("lr", 1e-4, 2e-3, log=True)

    # Print trial parameters at the start of each trial
    print("\n" + "="*70)
    print(f" Starting Trial {trial.number}")
    print(" Hyperparameters:")
    print(f" d_token = {d_Block}")
    print(f" n_blocks = {n_Blocks}")
    print(f" attention_n_heads = {Attention_n_heads}")
    print(f" ffn_d_hidden = {FFN_d_hidden}")
    print(f" residual_dropout = {Residual_droupout}")
    print(f" lr = {Lr}")
    print(f" n_freq = {n_frequencies}")
    print("="*70 + "\n")

    model = FTTransformer(
        n_cont_features=d_numerical,
        cat_cardinalities=cat_cardinalities,
        n_frequencies=n_frequencies,
        n_blocks=n_Blocks,
        d_block=d_Block,
        attention_n_heads=Attention_n_heads,
        ffn_d_hidden_multiplier=FFN_d_hidden,
        attention_dropout=Attention_dropout,
        ffn_dropout=FFN_dropout,
        residual_dropout=Residual_droupout,
        d_out=1 # output a single logit (binary)
    ).to(device)
    n_epochs = 2 if DEV_MODE else 100
    patience = 15

    model = torch.nn.DataParallel(model, device_ids = [0,1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=Lr * 5, # Recommended: 3–10× base LR
    steps_per_epoch=math.ceil(train_df_proc.shape[0] / BATCH_SIZE), # NUMBER OF BATCHES PER EPOCH
    epochs=n_epochs,
    pct_start=0.3,
    anneal_strategy="cos"
)

    loss_fn = F.binary_cross_entropy_with_logits

    def apply_model(batch: Dict[str, Tensor]) -> Tensor:
            return model(batch["X_cont"], batch.get("x_cat")).squeeze(-1)

    @torch.no_grad()
    def evaluate(part: str) -> float:
        model.eval()
        eval_batch_size = BATCH_SIZE
        y_pred = (
            torch.cat(
                [
                    apply_model(batch)
                    for batch in delu.iter_batches(data[part], eval_batch_size)
                ]
            )
            .cpu()
            .numpy()
        )
        y_true = data[part]["y"].cpu().numpy()
        y_pred = scipy.special.expit(y_pred)
        auc = roc_auc_score(y_true, y_pred)
        # y_pred = np.round(y_pred)
        # score = sklearn.metrics.accuracy_score(y_true, y_pred>0.3)
        return auc # The higher -- the better.


    print(f'Test score before training: AUC Val = {evaluate("val"):.4f}')


    batch_size = BATCH_SIZE
    epoch_size = math.ceil(train_df.shape[0] / batch_size)
    timer = delu.tools.Timer()
    early_stopping = delu.tools.EarlyStopping(patience, mode="max")
    best = {
        "val": -math.inf,
        "test": -math.inf,
        "epoch": -1,
    }

    print(f"Device: {device.type.upper()}")
    print("-" * 88 + "\n")
    timer.run()
    for epoch in range(n_epochs):
        for batch in tqdm(
            delu.iter_batches(data["train"], batch_size, shuffle=True),
            desc=f"Epoch {epoch}",
            total=epoch_size,
        ):
            model.train()
            optimizer.zero_grad()

            y_batch = batch["y"].float()
            weights = torch.where(
                y_batch == 1,
                torch.tensor(4.0, device=y_batch.device),
                torch.tensor(1.0, device=y_batch.device)
            )

            loss = loss_fn(apply_model(batch), batch["y"].float(), weight=weights)
            loss.backward()
            optimizer.step()
            scheduler.step()

        val_auc = evaluate("val")
        test_auc = evaluate("test")
        train_auc = evaluate('train')
        # print(f"(val) {val_score:.4f} (test) {test_score:.4f} [time] {timer}")
        print(f" AUC Train {train_auc:.4f} Val {val_auc:.4f} (test) {test_auc:.4f} [time] {timer}")

        early_stopping.update(val_auc)
        if early_stopping.should_stop():
            break

        if val_auc > best["val"]:
            print(" New best epoch! ")
            best = {"val": val_auc, "test": test_auc, "epoch": epoch, 'train':train_auc}
        print()

    trial_results.append({
        "trial_number": trial.number,
        "auc_val": val_auc,
        "auc_test": test_auc,
        "d_token": d_Block,
        "n_blocks": n_Blocks,
        "n_heads": Attention_n_heads,
        "ffn_d_hidden": FFN_d_hidden,
        "attention_dropout": Attention_dropout,
        "ffn_dropout": FFN_dropout,
        'Residual_droupout':Residual_droupout,
        'Lr':Lr,
        'n_frequencies':n_frequencies,
        'best':best,
        'time':timer.__str__(),
        'auc_train':train_auc
    })
    with open(str(RESULTS_DIR / "ftt_periodic_lc_trials.json"), "w") as f:
        json.dump(trial_results, f, indent=4)

    # print(trial_results[trial.number])
    print("\n\nResult:")
    print(best)
    return best['val']
study = optuna.create_study(direction="maximize")
study.optimize(optim, n_trials=50)